In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

In [14]:
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")

def tokenize(batch):
  return tokenizer(
      batch["text"],
      truncation=True,
      max_length=512,
  )

  dataset = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)

  data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [15]:
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto")

[transformers] If you want to use `RobertaLMHeadModel` as a standalone, add `is_decoder=True.`


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [16]:
training_args = TrainingArguments(
    output_dir="roberta-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"every_n_layers": 4},
    bf16=True,
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

ValueError: You should supply an encoding or a list of encodings to this method that includes input_ids, but you provided ['labels']